# Amazon review NLP

In [ ]:
import json
import pandas as pd
from tqdm import tqdm
import matplotlib.pyplot as plt
import re
import nltk
from afinn import Afinn

In [ ]:
json_lines = 0

with open('.data/Office_Products.jsonl', 'r') as fp:
    for line in fp:
        json_lines = json_lines + 1

In [ ]:
with open('.data/Office_Products.jsonl', 'r') as fp:
    for line in fp:
        test_dict = json.loads(line)
        break

In [ ]:
test_dict

In [ ]:
cols = ['rating', 'text', 'parent_asin']

In [ ]:
intermediate_batch = []
max_batch_size = 100000
intermediate_dfs = []

with open('.data/Office_Products.jsonl', 'r') as f:
    for line in tqdm(f, total=json_lines):
        r = json.loads(line)
        intermediate_batch.append({c: r.get(c) for c in cols})
        if len(intermediate_batch) == max_batch_size:
            intermediate_dfs.append(pd.DataFrame(intermediate_batch))
            intermediate_batch.clear()

if len(intermediate_batch) != 0:
    intermediate_dfs.append(pd.DataFrame(intermediate_batch))
    intermediate_batch.clear()

raw_reviews = pd.concat(intermediate_dfs, ignore_index=True)
intermediate_dfs.clear()

In [ ]:
raw_reviews = pd.read_csv('.data/raw_reviews.csv')

In [ ]:
raw_reviews.to_csv('.data/raw_reviews.csv', index=False)

In [ ]:
raw_reviews.info()

In [ ]:
raw_reviews.head()

In [ ]:
raw_reviews[raw_reviews.isna().any(axis=1)]

In [ ]:
i = 0
for index, text in raw_reviews.loc[~raw_reviews['text'].str.isalnum(), 'text'].items():
    if i == 10:
        break
    print(text)
    i = i + 1

In [ ]:
raw_reviews['text'] = raw_reviews['text'].str.replace(r'\[\[.*?\]\]', '', regex=True)

In [ ]:
raw_reviews['text'] = raw_reviews['text'].str.replace(r'\<.*?\>', '', regex=True)

In [ ]:
raw_reviews['rating'].value_counts().sort_index().plot.bar()

In [ ]:
raw_reviews = raw_reviews.loc[raw_reviews['rating'] != 0.0]

In [ ]:
raw_reviews['rating'] = (raw_reviews['rating'] - 3 ) / 2

In [ ]:
raw_reviews['rating'].value_counts().sort_index().plot.bar()

In [ ]:
review_counts = raw_reviews['parent_asin'].value_counts()

In [ ]:
valid_asins = review_counts[review_counts >= 10].index
raw_reviews = raw_reviews[raw_reviews['parent_asin'].isin(valid_asins)]

In [ ]:
mean_ratings = raw_reviews[['parent_asin', 'rating']].groupby('parent_asin').mean()

In [ ]:
mean_ratings.hist()

In [ ]:
tqdm.pandas()

In [ ]:
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer
lemmatizer = WordNetLemmatizer()

def lemmatize_string(text):
    if type(text) == str:
        tokens = word_tokenize(text)
        return [lemmatizer.lemmatize(word) for word in tokens]

In [ ]:
word_tokenize(raw_reviews.iloc[0].loc['text'])

In [ ]:
[lemmatizer.lemmatize(word) for word in word_tokenize(raw_reviews.iloc[0].loc['text'])]

In [ ]:
raw_reviews['lemmatized'] = raw_reviews['text'].progress_apply(lemmatize_string)

In [ ]:
raw_reviews['lemmatized']

In [ ]:
raw_reviews['text']

In [ ]:
english_stopwords = set(nltk.corpus.stopwords.words('english'))

def preprocess(text):
    text = text.lower()
    text = re.sub(r'[^a-z\s]', '', text)
    tokens = [word for word in text.split() if word not in english_stopwords]
    return tokens

In [ ]:
raw_reviews['tokens'] = raw_reviews['text'].progress_apply(preprocess)

In [ ]:
afinn = Afinn()

In [ ]:
def normalised_afinn(tokens):
    tokens_available = len(tokens)
    if tokens_available == 0:
        return
    else:
        return (afinn.score(' '.join(tokens)) / tokens_available) / 5

In [ ]:
raw_reviews['afinn_score'] = raw_reviews['tokens'].progress_apply(normalised_afinn)

In [ ]:
raw_reviews['afinn_score'].hist()

In [ ]:
from nltk.sentiment.vader import SentimentIntensityAnalyzer

In [ ]:
sia = SentimentIntensityAnalyzer()

In [ ]:
def vader_sentiment(tokens):
    return sia.polarity_scores(' '.join(tokens))['compound']

In [ ]:
raw_reviews['vader_score'] = raw_reviews['tokens'].progress_apply(vader_sentiment)

In [ ]:
raw_reviews[['rating', 'afinn_score', 'vader_score']].corr('spearman')

In [ ]:
asin_average = raw_reviews[['parent_asin', 'rating', 'afinn_score', 'vader_score']].groupby('parent_asin').mean()

In [ ]:
asin_average.corr()

In [ ]:
i = 0
for index, row in raw_reviews.sort_values(by=['rating', 'afinn_score', 'vader_score'], ascending=[True, False, False]).iterrows():
    if i == 100:
        break
    print(f'Scores\nRating: {row.loc['rating']}\nAfinn: {row.loc['afinn_score']}, Vader: {row.loc['vader_score']}')
    print(f'Verbatim: \"{row.loc['text']}\"')
    print(f'Tokens: {row.loc['tokens']}')
    print('')
    i = i + 1